<a href="https://colab.research.google.com/github/SharikaTasnim23/Thesis/blob/main/Shared%2BTower.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
"""
Wolfset Underwater Acoustic Dataset — Multi-Task Training Script
==================================================================
Architecture : Shared 1D CNN Backbone + 5 Task-Specific Towers
CV Strategy  : 4-fold StratifiedGroupKFold (grouped by Base_Session_ID,
               stratified by synchronous combo_label) + defensive session
               shifting for missing-state guardrails
Balancing    : Custom synchronous multi-task resampler (State-0 cap ~140,
               minority up-sampling to 140 via Gaussian jitter), applied
               STRICTLY inside the training fold after the split
Loss         : Sparse Categorical Focal Loss with per-head class weights
Notes        : This is being built EXACTLY per spec as a baseline run.
               This deliberately re-introduces architecture/loss/CV
               choices (shared backbone, focal loss, StratifiedGroupKFold)
               that earlier experiments on this project found inferior to
               a 5-separate-CNN architecture with plain SCCE and LOSO CV.
               Keep both result sets side by side for the comparison.
==================================================================
"""

'\nWolfset Underwater Acoustic Dataset — Multi-Task Training Script\n==================================================================\nArchitecture : Shared 1D CNN Backbone + 5 Task-Specific Towers\nCV Strategy  : 4-fold StratifiedGroupKFold (grouped by Base_Session_ID,\n               stratified by synchronous combo_label) + defensive session\n               shifting for missing-state guardrails\nBalancing    : Custom synchronous multi-task resampler (State-0 cap ~140,\n               minority up-sampling to 140 via Gaussian jitter), applied\n               STRICTLY inside the training fold after the split\nLoss         : Sparse Categorical Focal Loss with per-head class weights\nNotes        : This is being built EXACTLY per spec as a baseline run.\n               This deliberately re-introduces architecture/loss/CV\n               choices (shared backbone, focal loss, StratifiedGroupKFold)\n               that earlier experiments on this project found inferior to\n               a

In [ ]:
import os
import json
import numpy as np
import pandas as pd
import tensorflow as tf
from tensorflow.keras import layers, Model
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import EarlyStopping

from sklearn.model_selection import StratifiedGroupKFold
from sklearn.preprocessing import StandardScaler
from sklearn.utils.class_weight import compute_class_weight
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score

import matplotlib.pyplot as plt
import seaborn as sns

In [ ]:
# 0. Reproducibility
# ------------------------------------------------------------------
SEED = 42
np.random.seed(SEED)
tf.random.set_seed(SEED)
os.environ["PYTHONHASHSEED"] = str(SEED)

In [ ]:
# Cell 2: Global Constants and Paths
from google.colab import drive
drive.mount('/content/drive')


Mounted at /content/drive


In [ ]:
# 1. Global config
# ------------------------------------------------------------------
MOTOR_NAMES = [f"Motor_{i}" for i in range(1, 6)]
MOTOR_NUM_CLASSES = {
    "Motor_1": 6, "Motor_2": 6, "Motor_3": 6, "Motor_4": 6,  # 0..5
    "Motor_5": 5,                                            # 0..4
}
N_FOLDS = 4
TARGET_PER_STATE = 140          # balancing target used for both cap & floor
JITTER_STD = 0.01               # safe-jittering gaussian noise std
BATCH_SIZE = 32
EPOCHS = 150
PATIENCE = 15
FEATURE_DIM = 348

# Paths — adjust if your Drive layout differs from the earlier feature
# extraction notebook (Cell 5).
FEATURES_PATH = "/content/drive/MyDrive/Wolfset_Dataset/wolfset_348_features.npy"
METADATA_PATH = "/content/drive/MyDrive/Wolfset_Dataset/wolfset_segmented_metadata_348_features.csv"
OUTPUT_DIR = "/content/drive/MyDrive/Wolfset_Dataset/shared_backbone_Tower_results"
os.makedirs(OUTPUT_DIR, exist_ok=True)


In [ ]:
# 2. Load features + metadata produced by the feature-extraction notebook
# ------------------------------------------------------------------
def load_dataset():
    X = np.load(FEATURES_PATH).astype(np.float32)          # (N, 348, 1)
    if X.ndim == 2:
        X = X[..., np.newaxis]
    meta = pd.read_csv(METADATA_PATH)

    assert X.shape[0] == len(meta), (
        f"Feature/metadata row mismatch: X has {X.shape[0]} rows, "
        f"metadata has {len(meta)} rows."
    )
    assert X.shape[1] == FEATURE_DIM, f"Expected feature dim {FEATURE_DIM}, got {X.shape[1]}"

    y_dict = {m: meta[m].to_numpy().astype(np.int64) for m in MOTOR_NAMES}
    groups = meta["Base_Session_ID"].to_numpy()

    # combo_label: synchronous concatenation of all 5 motor states, used
    # purely as the StratifiedGroupKFold stratification target.
    combo_label = meta[MOTOR_NAMES].astype(str).agg("".join, axis=1).to_numpy()

    return X, y_dict, groups, combo_label, meta



In [ ]:
# 3. Custom synchronous multi-task resampler (train split ONLY)
# ------------------------------------------------------------------
def synchronous_resample(X_train, y_train_dict, target=TARGET_PER_STATE, jitter_std=JITTER_STD, rng=None):
    """
    Balances the training partition across all 5 motor heads simultaneously,
    operating on shared rows (each row carries 5 synchronous labels).

    Rules:
      - State 0 (Absent) down-sampled per motor to <= `target` rows.
      - Active states (1..5 depending on motor) with < `target` rows are
        up-sampled to exactly `target` via duplication + Gaussian jitter
        (mean=0, std=jitter_std) applied to the 1D feature vector only.
      - States already >= target are left untouched.

    Because a single row carries labels for all 5 motors at once, this is a
    best-effort *joint* balance, not 5 independent perfect balances:
      - Down-sampling: a row is only dropped for a motor's State-0 surplus
        if none of its OTHER motor labels are themselves an active state
        with a below-target count (protects other heads' minority rows).
      - Up-sampling: duplicating a row to fix one motor's minority state
        also duplicates its other 4 motor labels, so counts are re-checked
        sequentially motor-by-motor after each up-sampling pass.
    """
    if rng is None:
        rng = np.random.RandomState(SEED)

    X = X_train.copy()
    y = {m: arr.copy() for m, arr in y_train_dict.items()}
    n = X.shape[0]
    keep_mask = np.ones(n, dtype=bool)

    # ---- Pass 1: State-0 down-sampling (per motor, safety-checked) ----
    for motor in MOTOR_NAMES:
        zero_idx = np.where((y[motor] == 0) & keep_mask)[0]
        if len(zero_idx) <= target:
            continue

        # Rows that are "protected" because some OTHER motor's active state
        # at this row is currently under target (don't want to lose them).
        protected = np.zeros(len(zero_idx), dtype=bool)
        for other in MOTOR_NAMES:
            if other == motor:
                continue
            other_vals = y[other][zero_idx]
            for state in np.unique(other_vals):
                if state == 0:
                    continue
                count_state = np.sum((y[other] == state) & keep_mask)
                if count_state < target:
                    protected |= (other_vals == state)

        droppable = zero_idx[~protected]
        n_to_drop = len(zero_idx) - target
        if len(droppable) == 0:
            continue  # everything protected, cannot safely downsample further
        n_to_drop = min(n_to_drop, len(droppable))
        drop_choice = rng.choice(droppable, size=n_to_drop, replace=False)
        keep_mask[drop_choice] = False

    X = X[keep_mask]
    y = {m: arr[keep_mask] for m, arr in y.items()}

    # ---- Pass 2: minority up-sampling via safe jittering ----
    for motor in MOTOR_NAMES:
        n_classes = MOTOR_NUM_CLASSES[motor]
        for state in range(1, n_classes):  # active states only (skip 0)
            cur_idx = np.where(y[motor] == state)[0]
            cur_count = len(cur_idx)
            if cur_count == 0 or cur_count >= target:
                continue  # can't synthesize from nothing; document as gap
            n_needed = target - cur_count

            sampled_idx = rng.choice(cur_idx, size=n_needed, replace=True)
            X_new = X[sampled_idx].copy()
            noise = rng.normal(loc=0.0, scale=jitter_std, size=X_new.shape).astype(np.float32)
            X_new = X_new + noise

            y_new = {m: y[m][sampled_idx].copy() for m in MOTOR_NAMES}

            X = np.concatenate([X, X_new], axis=0)
            for m in MOTOR_NAMES:
                y[m] = np.concatenate([y[m], y_new[m]], axis=0)

    return X, y

In [ ]:
# 4. Defensive session shifting (missing-state guardrail)
# ------------------------------------------------------------------
def defensive_group_shift(train_idx, val_idx, y_dict, groups):
    """
    If any motor state is entirely absent from the training partition,
    move whole sessions (groups) from validation into training until that
    state is represented. Operates on RAW (pre-balancing) indices, since a
    state that never appears in raw train cannot be synthesized by
    up-sampling either.
    Returns updated (train_idx, val_idx).
    """
    train_idx = set(train_idx.tolist())
    val_idx = set(val_idx.tolist())

    changed = True
    guard = 0
    while changed and guard < 20:  # safety guard against infinite loop
        changed = False
        guard += 1
        for motor in MOTOR_NAMES:
            n_classes = MOTOR_NUM_CLASSES[motor]
            train_states = set(y_dict[motor][list(train_idx)].tolist())
            missing_states = set(range(n_classes)) - train_states
            if not missing_states:
                continue

            for state in missing_states:
                # find a val-side row with this missing state
                val_list = list(val_idx)
                if not val_list:
                    continue
                candidate_rows = [i for i in val_list if y_dict[motor][i] == state]
                if not candidate_rows:
                    continue  # state doesn't exist anywhere in val either
                donor_row = candidate_rows[0]
                donor_group = groups[donor_row]

                # shift the ENTIRE session (all rows sharing this group)
                session_rows = [i for i in val_list if groups[i] == donor_group]
                for r in session_rows:
                    val_idx.discard(r)
                    train_idx.add(r)
                changed = True

    return np.array(sorted(train_idx)), np.array(sorted(val_idx))



In [ ]:
# 5. Leakage assertions
# ------------------------------------------------------------------
def assert_no_leakage(X_train, X_val, train_idx, val_idx, groups):
    train_groups = set(groups[train_idx].tolist())
    val_groups = set(groups[val_idx].tolist())
    overlap = train_groups.intersection(val_groups)
    assert len(overlap) == 0, f"Session leakage detected! Overlapping groups: {overlap}"

    # duplicate raw-feature-row check (pre-jitter rows only, since jittered
    # rows are synthetic duplicates by design and won't collide with val)
    train_hashes = {hash(row.tobytes()) for row in X_train}
    val_hashes = {hash(row.tobytes()) for row in X_val}
    row_overlap = train_hashes.intersection(val_hashes)
    assert len(row_overlap) == 0, "Duplicate raw feature rows found across train/val boundary!"


In [ ]:
# 6. Model: Shared 1D CNN Backbone + 5 Task-Specific Towers
# ------------------------------------------------------------------
def build_shared_backbone_model():
    inp = layers.Input(shape=(FEATURE_DIM, 1), name="acoustic_input")

    x = inp
    for filters in (64, 128, 256):
        x = layers.Conv1D(filters, kernel_size=3, padding="same", activation="relu")(x)
        x = layers.BatchNormalization()(x)
        x = layers.MaxPooling1D(pool_size=2, padding="same")(x)

    shared_repr = layers.GlobalAveragePooling1D(name="shared_representation")(x)

    outputs = {}
    for motor in MOTOR_NAMES:
        t = layers.Dense(128, activation="relu")(shared_repr)
        t = layers.BatchNormalization()(t)
        t = layers.Dropout(0.4)(t)
        outputs[motor] = layers.Dense(
            MOTOR_NUM_CLASSES[motor], activation="softmax", name=motor
        )(t)

    model = Model(inputs=inp, outputs=outputs, name="Wolfset_SharedBackbone_MTL")
    return model


In [ ]:
# 7. Sparse Categorical Focal Loss with per-head class weights
# ------------------------------------------------------------------
def make_focal_loss(class_weights_vec, gamma=2.0):
    """
    class_weights_vec: 1D array/tensor of length n_classes for this head,
    computed from the (balanced) training distribution of that fold.
    """
    class_weights_vec = tf.constant(class_weights_vec, dtype=tf.float32)

    def loss_fn(y_true, y_pred):
        y_true = tf.cast(tf.reshape(y_true, [-1]), tf.int32)
        n_classes = tf.shape(y_pred)[-1]
        y_true_oh = tf.one_hot(y_true, depth=n_classes)

        eps = tf.keras.backend.epsilon()
        y_pred_clipped = tf.clip_by_value(y_pred, eps, 1.0 - eps)

        ce = -y_true_oh * tf.math.log(y_pred_clipped)
        modulating = tf.pow(1.0 - y_pred_clipped, gamma)
        fl = modulating * ce

        sample_weights = tf.gather(class_weights_vec, y_true)
        fl = tf.reduce_sum(fl, axis=-1) * sample_weights
        return fl

    return loss_fn


def compute_head_class_weights(y_train_balanced, n_classes):
    present_classes = np.unique(y_train_balanced)
    weights = compute_class_weight(
        class_weight="balanced", classes=present_classes, y=y_train_balanced
    )
    full_weights = np.ones(n_classes, dtype=np.float32)
    for cls, w in zip(present_classes, weights):
        full_weights[cls] = w
    return full_weights


In [ ]:
# 8. Sample-count logging (before/after balancing) & thesis-ready plots
# ------------------------------------------------------------------
def get_class_counts(y_arr, n_classes):
    """Returns a length-n_classes array of counts, indexed by state id."""
    counts = np.zeros(n_classes, dtype=int)
    vals, cnts = np.unique(y_arr, return_counts=True)
    for v, c in zip(vals, cnts):
        counts[int(v)] = c
    return counts


def print_balancing_summary(y_before, y_after, fold_idx):
    """
    Prints (and returns as a DataFrame) the per-motor, per-state sample
    count BEFORE (raw train split) and AFTER (post down/up-sampling)
    balancing — so you can see exactly what the resampler did.
    """
    print(f"\n--- Fold {fold_idx}: sample counts per state (raw -> balanced) ---")
    rows = []
    for motor in MOTOR_NAMES:
        n_classes = MOTOR_NUM_CLASSES[motor]
        before_counts = get_class_counts(y_before[motor], n_classes)
        after_counts = get_class_counts(y_after[motor], n_classes)
        for state in range(n_classes):
            rows.append({
                "Motor": motor,
                "State": state,
                "Before (raw train)": int(before_counts[state]),
                "After (balanced train)": int(after_counts[state]),
                "Delta": int(after_counts[state] - before_counts[state]),
            })
    df = pd.DataFrame(rows)
    print(df.to_string(index=False))
    return df


def plot_balancing_distribution(y_before, y_after, fold_idx, save_dir):
    """One figure, 5 subplots (one per motor): raw vs balanced counts per state."""
    fig, axes = plt.subplots(1, 5, figsize=(22, 4.2))
    for ax, motor in zip(axes, MOTOR_NAMES):
        n_classes = MOTOR_NUM_CLASSES[motor]
        before_counts = get_class_counts(y_before[motor], n_classes)
        after_counts = get_class_counts(y_after[motor], n_classes)
        x = np.arange(n_classes)
        width = 0.35
        ax.bar(x - width / 2, before_counts, width, label="Raw train", color="#94a3b8")
        ax.bar(x + width / 2, after_counts, width, label="Balanced train", color="#2563eb")
        ax.axhline(TARGET_PER_STATE, color="red", linestyle="--", linewidth=1)
        ax.set_xticks(x)
        ax.set_xticklabels([f"S{s}" for s in x])
        ax.set_title(motor)
        ax.set_ylabel("Sample count")
    axes[0].legend(loc="upper right", fontsize=8)
    fig.suptitle(f"Fold {fold_idx}: class distribution before vs after synchronous resampling", fontsize=13)
    plt.tight_layout()
    path = os.path.join(save_dir, f"fold_{fold_idx}_balancing_distribution.png")
    plt.savefig(path, dpi=150)
    plt.close()
    return path


def present_class_report(y_true, y_pred, motor_name, digits=4):
    """
    Classification report restricted to states that actually have support>0
    in y_true (i.e. states present in this validation set). Accuracy,
    macro avg, and weighted avg are therefore computed only over those
    present states — absent states never contaminate the averages.
    """
    y_true = np.asarray(y_true)
    y_pred = np.asarray(y_pred)
    present_classes = np.unique(y_true)  # states with support > 0
    target_names = [f"State {c}" for c in present_classes]

    report_str = classification_report(
        y_true, y_pred,
        labels=present_classes,
        target_names=target_names,
        zero_division=0,
        digits=digits,
    )
    report_dict = classification_report(
        y_true, y_pred,
        labels=present_classes,
        target_names=target_names,
        zero_division=0,
        output_dict=True,
    )
    return report_str, report_dict, present_classes


def plot_classification_report_heatmap(report_dict, motor_name, save_path):
    """
    Publication-style heatmap of precision/recall/f1-score per state, with
    support (n) shown in the row labels, and overall accuracy in the title.
    Suitable to drop straight into a thesis/paper figure or a PPT slide.
    """
    row_keys = [k for k in report_dict.keys() if k not in ("accuracy",)]
    df = pd.DataFrame(report_dict).T
    metric_df = df.loc[row_keys, ["precision", "recall", "f1-score"]].astype(float)
    support = df.loc[row_keys, "support"].astype(float)

    row_labels = [f"{k} (n={int(support[k])})" for k in row_keys]

    plt.figure(figsize=(6.5, max(3, 0.55 * len(row_keys) + 1.2)))
    ax = sns.heatmap(
        metric_df, annot=True, fmt=".2f", cmap="YlGnBu", vmin=0, vmax=1,
        cbar_kws={"label": "Score"}, linewidths=0.5, linecolor="white",
    )
    ax.set_yticklabels(row_labels, rotation=0)

    acc = report_dict.get("accuracy", None)
    title = f"{motor_name} — Classification Report (states with support>0 only)"
    if acc is not None:
        title += f"\nAccuracy = {acc:.3f}"
    plt.title(title, fontsize=11)
    plt.ylabel("")
    plt.tight_layout()
    plt.savefig(save_path, dpi=200)
    plt.close()


def plot_macro_weighted_summary(pooled_reports, save_path):
    """Headline figure: accuracy / macro-F1 / weighted-F1 across all 5 motors."""
    motors = list(pooled_reports.keys())

    # অত্যন্ত সেফ রিডিং লজিক (যাতে কোনো অবস্থাতেই KeyError না আসে)
    macro_f1 = [pooled_reports[m].get("macro avg", {}).get("f1-score", 0.0) for m in motors]
    weighted_f1 = [pooled_reports[m].get("weighted avg", {}).get("f1-score", 0.0) for m in motors]

    # এখানে .get() ব্যবহার করা হয়েছে, তাই ডিকশনারিতে 'accuracy' না থাকলেও কোড ক্র্যাশ করবে না
    acc = [pooled_reports[m].get("accuracy", 0.0) for m in motors]

    x = np.arange(len(motors))
    width = 0.25
    plt.figure(figsize=(9, 5))
    plt.bar(x - width, acc, width, label="Accuracy")
    plt.bar(x, macro_f1, width, label="Macro avg F1")
    plt.bar(x + width, weighted_f1, width, label="Weighted avg F1")
    plt.xticks(x, motors)
    plt.ylim(0, 1.05)
    plt.ylabel("Score")
    plt.title("Pooled performance summary across all 5 motor heads\n(states with support>0 only)")
    plt.legend()
    plt.tight_layout()
    plt.savefig(save_path, dpi=200)
    plt.close()


def plot_exact_match_across_folds(exact_match_scores, save_path):
    plt.figure(figsize=(6, 4))
    folds = [f"Fold {i+1}" for i in range(len(exact_match_scores))]
    bars = plt.bar(folds, exact_match_scores, color="#2563eb")
    plt.axhline(np.mean(exact_match_scores), color="red", linestyle="--",
                label=f"Mean = {np.mean(exact_match_scores):.3f}")
    for b, v in zip(bars, exact_match_scores):
        plt.text(b.get_x() + b.get_width() / 2, v + 0.01, f"{v:.3f}", ha="center", fontsize=9)
    plt.ylim(0, 1.05)
    plt.ylabel("Exact Match Accuracy")
    plt.title("Exact Match Accuracy per fold\n(all 5 motors correct simultaneously)")
    plt.legend()
    plt.tight_layout()
    plt.savefig(save_path, dpi=200)
    plt.close()


In [ ]:
# 9. Training / evaluation loop
# ------------------------------------------------------------------
def run_cv():
    X, y_dict, groups, combo_label, meta = load_dataset()

    sgkf = StratifiedGroupKFold(n_splits=N_FOLDS, shuffle=True, random_state=SEED)

    fold_reports = []
    exact_match_scores = []
    # aggregated predictions across all folds, per motor, for pooled reports
    agg_true = {m: [] for m in MOTOR_NAMES}
    agg_pred = {m: [] for m in MOTOR_NAMES}

    for fold_idx, (train_idx, val_idx) in enumerate(
        sgkf.split(X, combo_label, groups), start=1
    ):
        print(f"\n{'='*70}\nFOLD {fold_idx}/{N_FOLDS}\n{'='*70}")

        # ---- Defensive session shifting on RAW indices ----
        train_idx, val_idx = defensive_group_shift(train_idx, val_idx, y_dict, groups)

        X_train_raw = X[train_idx]
        X_val_raw = X[val_idx]
        y_train_raw = {m: y_dict[m][train_idx] for m in MOTOR_NAMES}
        y_val = {m: y_dict[m][val_idx] for m in MOTOR_NAMES}

        # ---- Leakage guardrails (pre-balancing, on raw split) ----
        assert_no_leakage(X_train_raw, X_val_raw, train_idx, val_idx, groups)

        # ---- Custom synchronous resampling (train ONLY) ----
        rng = np.random.RandomState(SEED + fold_idx)
        X_train_bal, y_train_bal = synchronous_resample(X_train_raw, y_train_raw, rng=rng)

        print(f"Train rows: raw={len(X_train_raw)} -> balanced={len(X_train_bal)} | Val rows: {len(X_val_raw)}")

        # ---- Sample-count logging: exactly what the resampler did ----
        dist_df = print_balancing_summary(y_train_raw, y_train_bal, fold_idx)
        dist_df.to_csv(os.path.join(OUTPUT_DIR, f"fold_{fold_idx}_balancing_counts.csv"), index=False)
        plot_balancing_distribution(y_train_raw, y_train_bal, fold_idx, OUTPUT_DIR)

        # ---- Train-only StandardScaler ----
        scaler = StandardScaler()
        n_train = X_train_bal.shape[0]
        X_train_flat = X_train_bal.reshape(n_train, -1)
        scaler.fit(X_train_flat)

        X_train_scaled = scaler.transform(X_train_flat).reshape(n_train, FEATURE_DIM, 1).astype(np.float32)
        n_val = X_val_raw.shape[0]
        X_val_scaled = scaler.transform(X_val_raw.reshape(n_val, -1)).reshape(n_val, FEATURE_DIM, 1).astype(np.float32)

        # ---- Model ----
        model = build_shared_backbone_model()

        losses = {}
        for motor in MOTOR_NAMES:
            cw = compute_head_class_weights(y_train_bal[motor], MOTOR_NUM_CLASSES[motor])
            losses[motor] = make_focal_loss(cw, gamma=2.0)

        model.compile(
            optimizer=Adam(learning_rate=0.001, clipnorm=1.0),
            loss=losses,
            metrics={m: "accuracy" for m in MOTOR_NAMES},
        )

        es = EarlyStopping(monitor="val_loss", patience=PATIENCE, restore_best_weights=True)

        y_train_targets = {m: y_train_bal[m] for m in MOTOR_NAMES}
        y_val_targets = {m: y_val[m] for m in MOTOR_NAMES}

        history = model.fit(
            X_train_scaled,
            y_train_targets,
            validation_data=(X_val_scaled, y_val_targets),
            epochs=EPOCHS,
            batch_size=BATCH_SIZE,
            callbacks=[es],
            verbose=2,
        )

        # ---- Evaluation ----
        preds = model.predict(X_val_scaled, verbose=0)
        # Keras returns a dict-like output when model has named dict outputs
        if isinstance(preds, dict):
            pred_labels = {m: np.argmax(preds[m], axis=1) for m in MOTOR_NAMES}
        else:
            # fallback if returned as list, in model.output_names order
            pred_labels = {
                m: np.argmax(preds[i], axis=1) for i, m in enumerate(model.output_names)
            }

        exact_match = np.mean(
            np.all(
                [pred_labels[m] == y_val[m] for m in MOTOR_NAMES], axis=0
            )
        )
        exact_match_scores.append(exact_match)
        print(f"Fold {fold_idx} Exact Match Accuracy: {exact_match:.4f}")

        fold_report = {}
        for motor in MOTOR_NAMES:
            report_str, report_dict, _ = present_class_report(y_val[motor], pred_labels[motor], motor)
            print(f"\n--- {motor} classification report (Fold {fold_idx}, states with support>0 only) ---")
            print(report_str)
            fold_report[motor] = report_dict

            agg_true[motor].extend(y_val[motor].tolist())
            agg_pred[motor].extend(pred_labels[motor].tolist())

        fold_reports.append(fold_report)

        with open(os.path.join(OUTPUT_DIR, f"fold_{fold_idx}_report.json"), "w") as f:
            json.dump(fold_report, f, indent=2)

    # ---- Aggregated (pooled) metrics across all folds ----
    print(f"\n{'='*70}\nPOOLED RESULTS ACROSS {N_FOLDS} FOLDS\n{'='*70}")
    print(f"Mean Exact Match Accuracy: {np.mean(exact_match_scores):.4f} (+/- {np.std(exact_match_scores):.4f})")

    pooled_reports = {}
    for motor in MOTOR_NAMES:
        report_str, report_dict, present_classes = present_class_report(
            agg_true[motor], agg_pred[motor], motor
        )
        print(f"\n--- Pooled classification report: {motor} (states with support>0 only) ---")
        print(report_str)
        pooled_reports[motor] = report_dict

        # Thesis/paper/PPT-ready heatmap of the report itself
        plot_classification_report_heatmap(
            report_dict, motor, os.path.join(OUTPUT_DIR, f"classification_report_heatmap_{motor}.png")
        )

        # Confusion matrix restricted to states that actually occur
        cm = confusion_matrix(agg_true[motor], agg_pred[motor], labels=present_classes)
        plt.figure(figsize=(6, 5))
        sns.heatmap(
            cm, annot=True, fmt="d", cmap="Blues",
            xticklabels=[f"S{c}" for c in present_classes],
            yticklabels=[f"S{c}" for c in present_classes],
        )
        plt.title(f"Pooled Confusion Matrix — {motor}")
        plt.xlabel("Predicted")
        plt.ylabel("True")
        plt.tight_layout()
        plt.savefig(os.path.join(OUTPUT_DIR, f"confusion_matrix_{motor}.png"), dpi=150)
        plt.close()

    # Headline figures for thesis / conference paper / presentation slides
    plot_macro_weighted_summary(pooled_reports, os.path.join(OUTPUT_DIR, "pooled_performance_summary.png"))
    plot_exact_match_across_folds(exact_match_scores, os.path.join(OUTPUT_DIR, "exact_match_accuracy_per_fold.png"))

    return {
        "fold_reports": fold_reports,
        "pooled_reports": pooled_reports,
        "exact_match_scores": exact_match_scores,
        "mean_exact_match": float(np.mean(exact_match_scores)),
    }


if __name__ == "__main__":
    results = run_cv()
    with open(os.path.join(OUTPUT_DIR, "summary.json"), "w") as f:
        json.dump(results, f, indent=2)
    print(f"\nAll results saved to: {OUTPUT_DIR}")



FOLD 1/4
Train rows: raw=1331 -> balanced=3159 | Val rows: 334

--- Fold 1: sample counts per state (raw -> balanced) ---
  Motor  State  Before (raw train)  After (balanced train)  Delta
Motor_1      0                1039                    2443   1404
Motor_1      1                  20                     140    120
Motor_1      2                 108                     156     48
Motor_1      3                  54                     140     86
Motor_1      4                  94                     140     46
Motor_1      5                  16                     140    124
Motor_2      0                1078                    2437   1359
Motor_2      1                  20                     140    120
Motor_2      2                  68                     162     94
Motor_2      3                  49                     140     91
Motor_2      4                  98                     140     42
Motor_2      5                  18                     140    122
Motor_3      0     

Fold 4 Exact Match Accuracy: 0.3391

--- Motor_1 classification report (Fold 4, states with support>0 only) ---
              precision    recall  f1-score   support

     State 0     0.9762    0.7765    0.8650       264
     State 2     0.4528    0.7500    0.5647        32
     State 4     0.9388    0.9388    0.9388        49

   micro avg     0.8814    0.7971    0.8371       345
   macro avg     0.7893    0.8218    0.7895       345
weighted avg     0.9223    0.7971    0.8476       345


--- Motor_2 classification report (Fold 4, states with support>0 only) ---
              precision    recall  f1-score   support

     State 0     0.7411    0.9745    0.8419       235
     State 2     0.3846    0.4167    0.4000        12
     State 4     0.7500    0.0306    0.0588        98

   micro avg     0.7270    0.6870    0.7064       345
   macro avg     0.6252    0.4739    0.4336       345
weighted avg     0.7312    0.6870    0.6041       345


--- Motor_3 classification report (Fold 4, states